# NEAT Data Model Inspection

Use [Cognite NEAT](https://docs.cognite.com/cdf/deploy/neat/index) to validate the oil & gas domain data model against 38+ rules covering AI readiness, connection integrity, CDF limits, query performance, and container/view consistency.

NEAT organizes validation into these categories:

| Category | Prefix | What it checks |
|----------|--------|----------------|
| **AI Readiness** | `NEAT-DMS-AI-READINESS` | Human-readable names and descriptions on models, views, properties, enumerations |
| **Connections** | `NEAT-DMS-CONNECTIONS` | Value types on direct relations, reverse connection validity |
| **Consistency** | `NEAT-DMS-CONSISTENCY` | Space/version alignment between views and data model |
| **Containers** | `NEAT-DMS-CONTAINER` | Referenced containers and properties exist, no constraint cycles |
| **Limits** | `NEAT-DMS-LIMITS` | Property counts, list sizes, view counts within CDF schema limits |
| **Performance** | `NEAT-DMS-PERFORMANCE` | Missing `requires` constraints, unindexed reverse relation targets |
| **Views** | `NEAT-DMS-VIEW` | Container mappings, implemented views exist, no cyclic implements |

Full rule reference: https://cognite-neat.readthedocs-hosted.com/en/latest/validation/

## Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials

load_dotenv(Path.cwd().parents[2] / ".env")

project = os.getenv("CDF_PROJECT")
cluster = os.getenv("CDF_CLUSTER")
client_id = os.getenv("CDF_CLIENT_ID") or os.getenv("IDP_CLIENT_ID")
client_secret = os.getenv("CDF_CLIENT_SECRET") or os.getenv("IDP_CLIENT_SECRET")
tenant_id = os.getenv("CDF_TENANT_ID") or os.getenv("IDP_TENANT_ID")
base_url = os.getenv("CDF_BASE_URL") or os.getenv("CDF_URL") or f"https://{cluster}.cognitedata.com"

credentials = OAuthClientCredentials(
    token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    client_id=client_id,
    client_secret=client_secret,
    scopes=[f"{base_url}/.default"],
)

client = CogniteClient(ClientConfig(
    client_name="neat-inspector",
    project=project,
    credentials=credentials,
    base_url=base_url,
))

C:\Users\JanIngeBergseth\AppData\Roaming\Python\Python313\site-packages\ipykernel\ipkernel.py:766: UserWarning: You are using version='7.87.0' of the SDK, however version='8.0.3' is available. To suppress this warning, either upgrade or do the following:
>>> from cognite.client.config import global_config
>>> global_config.disable_pypi_version_check = True
  _threading_Thread_run(self)


In [ ]:
%pip install cognite-neat==1.0.53

In [2]:
db_name = "cfihos_oil_and_gas"

# Delete all rows in every RAW table in the database, while keeping tables intact.
tables = client.raw.tables.list(db_name=db_name, limit=-1)

for table in tables:
    table_name = table.name
    rows = client.raw.rows.list(db_name=db_name, table_name=table_name, limit=-1)
    keys = [row.key for row in rows]

    if not keys:
        print(f"{table_name}: already empty")
        continue

    # RAW delete supports deleting by keys; do it in chunks for large tables.
    chunk_size = 1000
    for i in range(0, len(keys), chunk_size):
        client.raw.rows.delete(
            db_name=db_name,
            table_name=table_name,
            key=keys[i : i + chunk_size],
        )

    print(f"{table_name}: deleted {len(keys)} rows")

it_telecom_equipment: deleted 6 rows
tag: deleted 155 rows
functional_location: deleted 21 rows
turbine: deleted 6 rows
instrument_equipment: deleted 15 rows
equipment: deleted 70 rows
valve: deleted 12 rows
piping_pipeline: deleted 9 rows
timeseries: deleted 45 rows
subsea_equipment: deleted 9 rows
infrastructure: deleted 9 rows
mechanical_equipment: deleted 6 rows
compressor: deleted 21 rows
electrical_equipment: deleted 21 rows
maintenance_integrity: deleted 24 rows
file: deleted 22 rows
work_order: deleted 75 rows
miscellaneous_equipment: deleted 6 rows
enclosure: deleted 14 rows
notification: deleted 24 rows
work_order_operation: deleted 45 rows
tool: deleted 6 rows
failure_mode: deleted 56 rows
hse_equipment: deleted 9 rows
pump: deleted 9 rows
heat_exchanger: deleted 9 rows
drilling_equipment: deleted 14 rows


## Configure NeatSession

NEAT ships with predefined governance profiles that control which validators run:

| Profile | Description |
|---------|-------------|
| `legacy-additive` | Conservative — excludes AI readiness, some connection, and consistency checks |
| `deep-additive` | Comprehensive — enables **all** validators including AI readiness and performance |

The `deep-additive` profile is recommended for full coverage. Alpha flags below enable CDF-side analysis and experimental validators.

In [ ]:
from cognite.neat import NeatSession
from cognite.neat._config import NeatConfig


cfg = NeatConfig.create_predefined("deep-additive")

# Alpha features
cfg.alpha.enable_cdf_analysis = True
cfg.alpha.enable_solution_model_creation = True
cfg.alpha.enable_experimental_validators = True

neat = NeatSession(client, cfg)

In [ ]:
from cognite.neat import NeatSession, get_cognite_client
%load_ext autoreload
%autoreload 2

from cognite.neat._data_model.models.dms._references import ViewReference


## Read Data Model

`read.cdf(space, data_model_id, version)` fetches the deployed data model including all views, containers, and their inherited CDM/IDM dependencies. NEAT resolves the full graph so validation covers cross-space references (e.g., `cdf_cdm`, `cdf_idm`). The "Insights" count shows total findings; "errors" are blocking issues.

In [ ]:
neat.physical_data_model.read.cdf("dm_dom_oil_and_gas", "dm_oil_and_gas_domain_model", "v1")

## Inspect validation results

`neat.issues` returns all findings grouped by severity (errors, recommendations, auto-fixable). Results can be filtered by category, searched by keyword, and exported as CSV.

In [ ]:
neat.issues

## Model overview

`neat.physical_data_model` shows a summary of the loaded model: view count, container count, spaces, and node types.

In [ ]:
neat.physical_data_model